### Welcome! This notebook will help you get started with geospatial analysis using geopandas.

In [ ]:
# install geopandas if you haven't already
%pip install geopandas

In [ ]:
# import dependencies
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


Let's begin with reading and viewing the first dataset, the USDA Food Deserts shapefile. These GIS files contain data and metadata regarding all food desert census tracts in the state of Georgia. 

Geopandas uses very similar syntax to pandas for most functions, though there are some differences to be aware of. For example, geopandas exclusively uses the function read_file() rather than stating the file type.

In [ ]:
food_deserts = gpd.read_file("lila_halfmi_census.shp")
food_deserts.head()

As you can see above, geopandas "GeoDataFrames" can still use many standard functions of regular pandas DataFrames, such as .head(), .shape, .summarise(), etc.

In [ ]:
food_deserts.plot()

One important aspect of working with geospatial data is understanding which coordinate reference system (CRS) each dataset uses. This is similar to working with lengths in miles or kilometers or temperatures in Celsius or Fahrenheit; if you try to compare two datasets that were created using different measurement systems, the results will be nonsensical.

In [ ]:
food_deserts.crs

This CRS epsg:4326 is the most common CRS in all of GIS. This is named the "World Geodetic Standard" (WGS) because it is the global standard system to use. This coordinate reference system uses degrees latitude and longitude, though there is also a metric counterpart to the same world standard that can be used when more precise mathematical calculations are needed, epsg:3857

Now we will load another dataset, this time from excel. Because excel cannot store GIS metadata in the same way as shapefiles, we will first load the dataset as a regular DataFrame, then convert it to a GeoDataFrame.

In [ ]:
biz_data = pd.read_excel("Atlanta_Business_License_Records_2025.xlsx")
biz_data.head()

As you can see, the latitude and longitude values are stored in columns for every business address in the dataset. This enables you to create coordinates from those columns, using the geopandas function points_from_xy()

In [ ]:
biz_geo = gpd.GeoDataFrame(biz_data, geometry = gpd.points_from_xy(biz_data['longitude'], biz_data['latitude']), crs = 'epsg:4326')

In this one line, three important things are happening:
1. We coerce the dataframe to a geodataframe using geopandas
2. The latitude and longitude columns are converted to coordinate points using points_from_xy()
3. Because there are no metadata files geopandas can use to determine the CRS, you must manually state which CRS was used in these coordinates. In this case, we know that the datset was created using a geocoding API that returns coordinates in epsg:4326, which is extremely common. If you ever don't know which CRS was used for a dataset you want to work with, there are ways to troubleshoot, but it's best to download geospatial data as shapefiles or geojson whenever possible to avoid the issue.

In [ ]:
biz_geo.plot()

In [ ]:
atl = gpd.read_file("Atlanta_City_Limits.shp")
atl.plot()

In [ ]:
atl.crs

In [ ]:
biz_geo = biz_geo.to_crs(crs = 'epsg:3857')

In [ ]:
biz_geo.crs == atl.crs

In [ ]:
biz_check = gpd.sjoin(biz_geo, atl, how = 'left', predicate = 'within')
atl_biz = biz_check.loc[(biz_check['NAME'] == 'Atlanta')]
atl_biz.shape

In [ ]:
atl_biz.plot()

In [ ]:
# buffer example
biz_halfmi = atl_biz.geometry.buffer(804.5, resolution = 20, cap_style = 'round')
biz_halfmi.plot()